# Черновой notebook для VS Code → Colab

Этот ноутбук нужен, чтобы быстро открыть `.ipynb` в VS Code и затем запускать его в Google Colab с GPU.

## 1) Базовая инициализация

Запустите ячейки по порядку сверху вниз.

Если вы открыли ноутбук в Colab, сначала подключите GPU runtime.
Если вы запускаете в VS Code локально, убедитесь, что активировано нужное Python-окружение.

In [ ]:
import os
import sys
from pathlib import Path

# Определяем корень проекта
if Path('/content').exists():
    # Colab: ожидаем, что вы загрузили/клонировали репозиторий
    # При необходимости замените путь ниже на ваш
    CANDIDATES = [Path('/content/T-bank'), Path('/content/repo'), Path('/content/project')]
    project_root = next((p for p in CANDIDATES if p.exists()), Path.cwd())
else:
    project_root = Path.cwd()

os.chdir(project_root)
print('Project root:', project_root)
print('Python:', sys.version)
print('Executable:', sys.executable)

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
else:
    print('GPU не найден. Для обучения нужен Colab GPU runtime.')

## 2) Установка зависимостей

Эта ячейка установит пакеты из `requirements.txt`.

Если запускаете в чистом Colab, можно дополнительно обновить `pip`.

In [ ]:
import subprocess
import sys
from pathlib import Path

req = Path('requirements.txt')
if not req.exists():
    raise FileNotFoundError('requirements.txt не найден в корне проекта')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=True)
print('Dependencies installed ✅')

## 3) Подготовка NanoVLM и каталогов артефактов

Склонируем `huggingface/nanoVLM` в `external/nanoVLM` (если его ещё нет) и создадим папки с результатами.

In [ ]:
import subprocess
from pathlib import Path

nanovlm_dir = Path('external/nanoVLM')
nanovlm_dir.parent.mkdir(parents=True, exist_ok=True)

if not nanovlm_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/huggingface/nanoVLM.git', str(nanovlm_dir)], check=True)
    print('Cloned nanoVLM ->', nanovlm_dir)
else:
    print('nanoVLM already exists ->', nanovlm_dir)

for path in ['artifacts/datasets', 'artifacts/sft', 'artifacts/grpo_action', 'artifacts/grpo_text_action', 'artifacts/plots']:
    Path(path).mkdir(parents=True, exist_ok=True)

print('Project dirs prepared ✅')

## 4) Запуск пайплайна (SFT + GRPO + сравнение)

Ниже можно выбрать профиль:
- `mvp` — короткий быстрый прогон;
- `full` — длиннее, ближе к финальному эксперименту.

После завершения появятся:
- `artifacts/*/history.csv`
- `artifacts/plots/success_rate.png`
- `artifacts/plots/summary_table.csv`

In [ ]:
import subprocess
import sys

RUN_PROFILE = 'mvp'  # поменяйте на 'full' для более длинного прогона

if RUN_PROFILE == 'mvp':
    cmds = [
        [sys.executable, '-m', 'src.data.generate_expert_dataset', '--out_dir', 'artifacts/datasets', '--train_episodes', '4000', '--val_episodes', '500', '--test_episodes', '500', '--seed', '42'],
        [sys.executable, '-m', 'src.train.train_sft', '--train_npz', 'artifacts/datasets/train.npz', '--val_npz', 'artifacts/datasets/val.npz', '--mode', 'action', '--epochs', '2', '--output_dir', 'artifacts/sft', '--model_source', 'lusxvr/nanoVLM-450M', '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.train.train_grpo', '--init_checkpoint', 'artifacts/sft/checkpoint_last', '--mode', 'action', '--updates', '100', '--episodes_per_update', '4', '--group_size', '4', '--output_dir', 'artifacts/grpo_action', '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.train.train_grpo', '--init_checkpoint', 'artifacts/sft/checkpoint_last', '--mode', 'text_action', '--updates', '100', '--episodes_per_update', '4', '--group_size', '4', '--output_dir', 'artifacts/grpo_text_action', '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.eval.compare_runs', '--sft', 'artifacts/sft/history.csv', '--grpo_action', 'artifacts/grpo_action/history.csv', '--grpo_text_action', 'artifacts/grpo_text_action/history.csv', '--out_dir', 'artifacts/plots'],
    ]
else:
    cmds = [
        [sys.executable, '-m', 'src.data.generate_expert_dataset', '--out_dir', 'artifacts/datasets', '--train_episodes', '20000', '--val_episodes', '2000', '--test_episodes', '2000', '--seed', '42'],
        [sys.executable, '-m', 'src.train.train_sft', '--train_npz', 'artifacts/datasets/train.npz', '--val_npz', 'artifacts/datasets/val.npz', '--mode', 'action', '--epochs', '5', '--output_dir', 'artifacts/sft', '--model_source', 'lusxvr/nanoVLM-450M', '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.train.train_grpo', '--init_checkpoint', 'artifacts/sft/checkpoint_last', '--mode', 'action', '--updates', '300', '--episodes_per_update', '8', '--group_size', '8', '--output_dir', 'artifacts/grpo_action', '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.train.train_grpo', '--init_checkpoint', 'artifacts/sft/checkpoint_last', '--mode', 'text_action', '--updates', '300', '--episodes_per_update', '8', '--group_size', '8', '--output_dir', 'artifacts/grpo_text_action', '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.eval.compare_runs', '--sft', 'artifacts/sft/history.csv', '--grpo_action', 'artifacts/grpo_action/history.csv', '--grpo_text_action', 'artifacts/grpo_text_action/history.csv', '--out_dir', 'artifacts/plots'],
    ]

for cmd in cmds:
    print('\n>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('\nPipeline finished ✅')
print('See artifacts/plots/success_rate.png and artifacts/plots/summary_table.csv')

## 5) Просмотр результатов

Проверьте, что все ключевые артефакты созданы после обучения.

In [ ]:
from pathlib import Path
import pandas as pd

required = [
    'artifacts/sft/history.csv',
    'artifacts/grpo_action/history.csv',
    'artifacts/grpo_text_action/history.csv',
    'artifacts/plots/summary_table.csv',
    'artifacts/plots/success_rate.png',
]

for path in required:
    p = Path(path)
    print(f'{path}:', 'OK' if p.exists() else 'MISSING')

summary_path = Path('artifacts/plots/summary_table.csv')
if summary_path.exists():
    display(pd.read_csv(summary_path))

## 6) Публикация в GitHub

Заполните `YOUR_REPO_URL` и запустите ячейку ниже.

Совет: перед пушем убедитесь, что большие бинарники не попадают в git (модели/чекпоинты лучше хранить как артефакты вне репозитория).

In [ ]:
import subprocess

YOUR_REPO_URL = 'https://github.com/SergeySolovyev/T-Lab-2026.-Multimodal-VLMs.git'

commands = [
    ['git', 'status'],
    ['git', 'add', '.'],
    ['git', 'commit', '-m', 'MiniGrid EmptyEnv + NanoVLM SFT/GRPO pipeline'],
    ['git', 'branch', '-M', 'main'],
    ['git', 'remote', 'remove', 'origin'],
    ['git', 'remote', 'add', 'origin', YOUR_REPO_URL],
    ['git', 'push', '-u', 'origin', 'main'],
]

for cmd in commands:
    print('>>>', ' '.join(cmd))
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

In [ ]:
import subprocess
import sys
from pathlib import Path

SEEDS = [11, 22, 33, 44, 55]  # можно расширить
RUN_PROFILE = 'mvp'  # 'mvp' | 'full'
BASE_DIR = Path('artifacts/multiseed')
BASE_DIR.mkdir(parents=True, exist_ok=True)

if RUN_PROFILE == 'mvp':
    train_episodes, val_episodes, test_episodes = 4000, 500, 500
    sft_epochs = 2
    grpo_updates, episodes_per_update, group_size = 100, 4, 4
else:
    train_episodes, val_episodes, test_episodes = 20000, 2000, 2000
    sft_epochs = 5
    grpo_updates, episodes_per_update, group_size = 300, 8, 8

for seed in SEEDS:
    seed_dir = BASE_DIR / f'seed_{seed}'
    ds_dir = seed_dir / 'datasets'
    sft_dir = seed_dir / 'sft'
    ga_dir = seed_dir / 'grpo_action'
    gt_dir = seed_dir / 'grpo_text_action'
    plots_dir = seed_dir / 'plots'

    for p in [ds_dir, sft_dir, ga_dir, gt_dir, plots_dir]:
        p.mkdir(parents=True, exist_ok=True)

    cmds = [
        [sys.executable, '-m', 'src.data.generate_expert_dataset', '--out_dir', str(ds_dir), '--train_episodes', str(train_episodes), '--val_episodes', str(val_episodes), '--test_episodes', str(test_episodes), '--seed', str(seed)],
        [sys.executable, '-m', 'src.train.train_sft', '--train_npz', str(ds_dir / 'train.npz'), '--val_npz', str(ds_dir / 'val.npz'), '--mode', 'action', '--epochs', str(sft_epochs), '--seed', str(seed), '--output_dir', str(sft_dir), '--model_source', 'lusxvr/nanoVLM-450M', '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.train.train_grpo', '--init_checkpoint', str(sft_dir / 'checkpoint_last'), '--mode', 'action', '--updates', str(grpo_updates), '--episodes_per_update', str(episodes_per_update), '--group_size', str(group_size), '--seed', str(seed), '--output_dir', str(ga_dir), '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.train.train_grpo', '--init_checkpoint', str(sft_dir / 'checkpoint_last'), '--mode', 'text_action', '--updates', str(grpo_updates), '--episodes_per_update', str(episodes_per_update), '--group_size', str(group_size), '--seed', str(seed), '--output_dir', str(gt_dir), '--nanovlm_repo', 'external/nanoVLM'],
        [sys.executable, '-m', 'src.eval.compare_runs', '--sft', str(sft_dir / 'history.csv'), '--grpo_action', str(ga_dir / 'history.csv'), '--grpo_text_action', str(gt_dir / 'history.csv'), '--out_dir', str(plots_dir)],
    ]

    print(f'\n===== SEED {seed} =====')
    for cmd in cmds:
        print('>>>', ' '.join(cmd))
        subprocess.run(cmd, check=True)

print('\nMulti-seed runs finished ✅')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path('artifacts/multiseed')
OUT_DIR = Path('artifacts/multiseed_aggregate')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUCCESS_THRESHOLD = 0.80
MAX_STEPS = 100
RUN_PROFILE = 'mvp'  # должен совпадать с multi-seed ячейкой

if RUN_PROFILE == 'mvp':
    episodes_per_update, group_size = 4, 4
else:
    episodes_per_update, group_size = 8, 8

def load_method_histories(method_subdir):
    result = {}
    for seed_dir in sorted(BASE_DIR.glob('seed_*')):
        seed = int(seed_dir.name.split('_')[-1])
        path = seed_dir / method_subdir / 'history.csv'
        if path.exists():
            result[seed] = pd.read_csv(path)
    return result

def first_reach_progress(df, threshold):
    if 'update' in df.columns:
        xcol = 'update'
    elif 'epoch' in df.columns:
        xcol = 'epoch'
    else:
        return np.nan
    hit = df[df['success_rate'] >= threshold]
    if len(hit) == 0:
        return np.nan
    return float(hit.iloc[0][xcol])

def summarize_final(histories):
    finals = []
    for seed, df in histories.items():
        finals.append({'seed': seed, 'success_rate': float(df['success_rate'].iloc[-1]), 'avg_return': float(df['avg_return'].iloc[-1])})
    out = pd.DataFrame(finals)
    return out

def align_mean_std(histories, xcol):
    all_x = sorted(set(np.concatenate([df[xcol].values for df in histories.values()])))
    curves = []
    for seed, df in histories.items():
        s = pd.Series(df['success_rate'].values, index=df[xcol].values)
        s = s.reindex(all_x).interpolate(method='linear').ffill().bfill()
        curves.append(s.values)
    arr = np.array(curves)
    return np.array(all_x), arr.mean(axis=0), arr.std(axis=0)

sft_h = load_method_histories('sft')
ga_h = load_method_histories('grpo_action')
gt_h = load_method_histories('grpo_text_action')

if not (sft_h and ga_h and gt_h):
    raise RuntimeError('Не найдены все history.csv для multiseed. Сначала выполните секцию 7.')

sft_final = summarize_final(sft_h)
ga_final = summarize_final(ga_h)
gt_final = summarize_final(gt_h)

rows = []
for name, df in [('SFT', sft_final), ('GRPO-action', ga_final), ('GRPO-text+action', gt_final)]:
    rows.append({
        'method': name,
        'n_seeds': int(len(df)),
        'final_success_mean': float(df['success_rate'].mean()),
        'final_success_std': float(df['success_rate'].std(ddof=1)) if len(df) > 1 else 0.0,
        'final_return_mean': float(df['avg_return'].mean()),
        'final_return_std': float(df['avg_return'].std(ddof=1)) if len(df) > 1 else 0.0,
    })

summary = pd.DataFrame(rows)

# sample efficiency только для GRPO-методов (по update)
def sample_efficiency_table(histories, method_name):
    rec = []
    for seed, df in histories.items():
        upd = first_reach_progress(df, SUCCESS_THRESHOLD)
        if np.isnan(upd):
            rec.append({'method': method_name, 'seed': seed, 'update_to_threshold': np.nan, 'episodes_to_threshold': np.nan, 'env_steps_to_threshold': np.nan})
        else:
            episodes = upd * episodes_per_update * group_size
            env_steps = episodes * MAX_STEPS
            rec.append({'method': method_name, 'seed': seed, 'update_to_threshold': upd, 'episodes_to_threshold': episodes, 'env_steps_to_threshold': env_steps})
    return pd.DataFrame(rec)

se_ga = sample_efficiency_table(ga_h, 'GRPO-action')
se_gt = sample_efficiency_table(gt_h, 'GRPO-text+action')
se_all = pd.concat([se_ga, se_gt], ignore_index=True)

se_summary = se_all.groupby('method', dropna=False).agg(
    seeds=('seed', 'count'),
    reached=('update_to_threshold', lambda x: int(np.sum(~np.isnan(x)))),
    update_mean=('update_to_threshold', 'mean'),
    update_std=('update_to_threshold', 'std'),
    episodes_mean=('episodes_to_threshold', 'mean'),
    env_steps_mean=('env_steps_to_threshold', 'mean'),
).reset_index()

# агрегированный график success rate для двух GRPO-режимов
x_ga, m_ga, s_ga = align_mean_std(ga_h, 'update')
x_gt, m_gt, s_gt = align_mean_std(gt_h, 'update')

plt.figure(figsize=(8,5))
plt.plot(x_ga, m_ga, label='GRPO-action')
plt.fill_between(x_ga, m_ga - s_ga, m_ga + s_ga, alpha=0.2)
plt.plot(x_gt, m_gt, label='GRPO-text+action')
plt.fill_between(x_gt, m_gt - s_gt, m_gt + s_gt, alpha=0.2)
plt.axhline(SUCCESS_THRESHOLD, linestyle='--', linewidth=1, label=f'threshold={SUCCESS_THRESHOLD:.2f}')
plt.xlabel('Update')
plt.ylabel('Success rate')
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'grpo_success_mean_std.png', dpi=180)

summary.to_csv(OUT_DIR / 'final_metrics_mean_std.csv', index=False)
se_all.to_csv(OUT_DIR / 'sample_efficiency_by_seed.csv', index=False)
se_summary.to_csv(OUT_DIR / 'sample_efficiency_summary.csv', index=False)

print('Saved to:', OUT_DIR)
print('\nFinal metrics (mean±std):')
display(summary)
print('\nSample efficiency summary:')
display(se_summary)

## 8) Агрегация по seeds + sample efficiency

Считает финальные `mean±std` и sample efficiency до порога `SUCCESS_THRESHOLD`.

Для GRPO sample efficiency оценивается в env-steps и эпизодах до достижения порога.

## 7) Multi-seed запуск (5+ seeds)

Запускает серию экспериментов и сохраняет результаты в `artifacts/multiseed/seed_<N>/...`.

Для экономии времени можно сначала протестировать на 2 seeds, потом вернуть 5+.